In [7]:
# =========================================
# 🎬 MOVIE RATING PREDICTION (FINAL)
# =========================================

import pandas as pd
import numpy as np
import pickle
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# =========================================
# LOAD DATA
# =========================================

df = pd.read_csv("../data/movie.csv")

print("Dataset Loaded")
print(df.head())
print(df.columns)

# =========================================
# CLEANING
# =========================================

df = df.dropna(subset=['Rating'])

df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')
df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce')

df.fillna(0, inplace=True)

# =========================================
# FEATURE ENGINEERING ⭐
# =========================================

df['combined'] = (
    df['Genre'].astype(str) + " " +
    df['Director'].astype(str) + " " +
    df['Actor1'].astype(str) + " " +
    df['Actor2'].astype(str) + " " +
    df['Actor3'].astype(str)
)

# NLP vectorization
cv = CountVectorizer(max_features=1000)
X_text = cv.fit_transform(df['combined']).toarray()

# Numeric features
X_num = df[['Votes', 'Duration']].values

# Combine features
X = np.hstack((X_text, X_num))

y = df['Rating']

# =========================================
# TRAIN TEST SPLIT
# =========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# MODEL
# =========================================

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

# =========================================
# EVALUATION
# =========================================

y_pred = model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))

# =========================================
# SAVE MODEL
# =========================================

os.makedirs("../model", exist_ok=True)

pickle.dump(model, open("../model/movie_model.pkl", "wb"))
pickle.dump(cv, open("../model/vectorizer.pkl", "wb"))

print("✅ Model Saved Successfully!")

Dataset Loaded
     Name  Year  Duration                 Genre  Rating  Votes  \
0  Movie1  2019       109                 Drama     7.0    800   
1  Movie2  2021        90         Drama Musical     6.5    500   
2  Movie3  2019       110        Comedy Romance     4.4    350   
3  Movie4  2010       105                 Drama     7.2   1200   
4  Movie5  1997       147  Comedy Drama Musical     4.7    827   

             Director        Actor1              Actor2           Actor3  
0       Gaurav Bakshi  Rasika Dugal      Vivek Ghamande    Arvind Jangid  
1  Soumyajit Majumdar  Sayani Gupta   Plabita Borthakur       Roy Angana  
2          Ovais Khan       Prateik          Ishita Raj  Siddhant Kapoor  
3        Amol Palekar  Rajat Kapoor  Rituparna Sengupta      Antara Mali  
4        Rahul Rawail    Bobby Deol       Aishwarya Rai    Shammi Kapoor  
Index(['Name', 'Year', 'Duration', 'Genre', 'Rating', 'Votes', 'Director',
       'Actor1', 'Actor2', 'Actor3'],
      dtype='object')
MSE